# Phase-graph cyclicity and the coherence score

This notebook writes the coherence detector as a statistic of the same phase-transition graph used by S2G, then implements that statistic and runs it on one recording (the same file as the example cell in `example.ipynb`).

## 0. The raw data

Fix a frequency bin. After quantization you have a sequence of integers

$$
q(1),q(2),\ldots,q(T)\in\{0,1,\ldots,Q-1\}.
$$

A **transition** is one consecutive pair $(q(t),q(t+1))$. There are $T-1$ of them.

Write

$$
N_{ij}=\text{number of }t\text{ with }q(t)=i\text{ and }q(t+1)=j,
\qquad
N_i=\sum_j N_{ij}.
$$

$N_i$ is how often the sequence is in state $i$ and then takes a step. $N_{ij}$ is how often that step goes to $j$.

## 1. Two ways to turn counts into weights

**Row weight** (this is $w_{ij}$ in the paper):

$$
w_{ij}=\frac{N_{ij}}{N_i}
\qquad\text{(if $N_i=0$, set the whole row to $0$)}.
$$

Of the times we left $i$, what fraction went to $j$? For each $i$ with $N_i>0$, $\sum_j w_{ij}=1$.

**Joint weight:**

$$
M_{ij}=\frac{N_{ij}}{T-1}.
$$

Of all transitions in the recording, what fraction were $i$ then $j$? Now $\sum_{i,j}M_{ij}=1$.

These are not the same:

$$
M_{ij}=\frac{N_i}{T-1}\,w_{ij}.
$$

The factor $N_i/(T-1)$ is the fraction of all steps that *started* in $i$. If every state is used equally often as a start, then $N_i=(T-1)/Q$ for every $i$, and $M_{ij}=w_{ij}/Q$. Both lemmas in the paper are in that equal-start situation (in expectation). Until stated otherwise, assume that, so the two weights differ only by the constant $1/Q$.

## 2. Jump size

The states sit on a circle: after $Q-1$ comes $0$.

If a transition is $i\to j$, its **jump size** is the unique $d\in\{0,\ldots,Q-1\}$ with

$$
j\equiv i+d\pmod Q.
$$

Examples for $Q=4$: $0\to 1$ has $d=1$; $3\to 0$ has $d=1$; $0\to 0$ has $d=0$; $0\to 3$ has $d=3$.

Define

$$
p(d)=\sum_{i=0}^{Q-1}M_{i,\,(i+d)\bmod Q}.
$$

$p(d)$ is the fraction of all transitions that have jump size $d$, and $\sum_d p(d)=1$.

If starts are equally common,

$$
p(d)=\frac{1}{Q}\sum_{i=0}^{Q-1}w_{i,\,(i+d)\bmod Q}.
$$

## 3. What $C_r(G)$ does

The paper's formula is

$$
C_r(G)=\max_{\Delta\in\{0,\ldots,Q-1\}}
\frac{1}{Q}\sum_{i=0}^{Q-1}
\sum_{j:\,d_Q\bigl(j,\,(i+\Delta)\bmod Q\bigr)\le r}
w_{ij}.
$$

**Circular distance.** For two states $a,b$,

$$
d_Q(a,b)=\min\bigl(|a-b|,\,Q-|a-b|\bigr).
$$

It is the shorter number of steps between them on the circle. For $Q=4$: $d_Q(0,1)=1$, $d_Q(0,3)=1$, $d_Q(0,2)=2$.

**The procedure, in order.**

1. Pick a candidate jump $\Delta$ (try every value $0,1,\ldots,Q-1$).
2. For each start state $i$, the ideal next state if every jump were exactly $\Delta$ is $(i+\Delta)\bmod Q$.
3. From $i$, add $w_{ij}$ only for those $j$ whose circular distance to that ideal next state is at most $r$.
4. Average those sums over the $Q$ start states (that is the $1/Q$).
5. Keep the $\Delta$ that gives the largest average. That number is $C_r(G)$.

So $C_r$ is: choose the best single target jump $\Delta$, then credit every transition whose jump is within $r$ steps of $\Delta$, averaged evenly over start states.

## 4. The special case $r=0$

If $r=0$, "$j$ within distance $0$ of the ideal next state" means $j$ must be exactly $(i+\Delta)\bmod Q$. Then

$$
C_0(G)=\max_{\Delta}\frac{1}{Q}\sum_{i}w_{i,\,(i+\Delta)\bmod Q}.
$$

If starts are equally common, the quantity being maximized is exactly $p(\Delta)$, so

$$
C_0(G)=\max_{d}\,p(d).
$$

- **Lemma 2 (pure tone).** Every transition has the same jump $k$. Then $p(k)=1$ and $p(d)=0$ for $d\neq k$. Hence $C_0=1$.
- **Lemma 1 (white noise).** From every $i$, each next state is equally likely, so $w_{ij}=1/Q$. Then $p(d)=1/Q$ for all $d$. Hence $C_0=1/Q$.

## 5. What $r>0$ adds

For $r=1$ and a fixed $\Delta$, you do not only count jump $\Delta$. You also count jumps whose circular distance to $\Delta$ is $1$.

Example, $Q=4$, $\Delta=1$: you count $d=1$, and also $d=0$ and $d=2$. You do not count $d=3$, because $d_Q(3,1)=2>1$.

If starts are equally common,

$$
C_r(G)=\max_{\Delta}\sum_{d:\,d_Q(d,\Delta)\le r}p(d).
$$

Add a few neighboring bins of $p$, take the placement of that block that gives the largest sum. That is all $r$ does: a tolerance on how exact the jump must be.

**Numbers, $Q=4$.** Tone sequence $0,1,2,3,0,1,2,3,0$ (eight transitions): every jump has $d=1$, so $p(1)=1$, $C_0=1$, and $C_1=1$. Noise-like: two jumps of each size $0,1,2,3$ gives $p(d)=1/4$, $C_0=1/4$, and $C_1=3/4$ (three neighboring bins).

## 6. The coherence number, from the same $p$

Do **not** search over $\Delta$ and do **not** choose an $r$.

For each jump size $d$, take the point on the unit circle at angle $2\pi d/Q$:

$$
\bigl(\cos(2\pi d/Q),\;\sin(2\pi d/Q)\bigr),
$$

which is the complex number $e^{j 2\pi d/Q}$. Scale it by $p(d)$. Add the $Q$ arrows as vectors. $C_{\mathrm{coh}}$ is the **length** of the sum:

$$
C_{\mathrm{coh}}(G)
=\Biggl|\sum_{d=0}^{Q-1}p(d)\,e^{j 2\pi d/Q}\Biggr|.
$$

- All mass on one $d$: one arrow of length $1$, sum has length $1$.
- Equal mass $1/Q$ on all $Q$ directions: arrows cancel, length $0$.

Same $p$ as in §2. Different question: $C_r$ asks whether there is a $\Delta$ such that most jumps are near $\Delta$. $C_{\mathrm{coh}}$ asks whether the arrows for those jumps point the same way.

## 7. Why the two sums for $C_{\mathrm{coh}}$ are the same

Write $\theta_i=2\pi i/Q$. For a transition $i\to j$ with jump $d$, $j\equiv i+d\pmod Q$, so $\theta_j-\theta_i=2\pi d/Q$ and $e^{j(\theta_j-\theta_i)}=e^{j 2\pi d/Q}$.

$$
\sum_d p(d)\,e^{j 2\pi d/Q}
=\sum_d\sum_i M_{i,\,(i+d)\bmod Q}\,e^{j 2\pi d/Q}
=\sum_{i,j}M_{ij}\,e^{j(\theta_j-\theta_i)}.
$$

The pairs $(i,\,(i+d)\bmod Q)$ as $i$ and $d$ run through all values are exactly all pairs $(i,j)$. One sum is by jump size, the other is by edge.

## 8. From that sum to the implemented score

Replace the discrete angle $\theta_{q(t)}$ by the STFT phase $\phi(t)$, and the average over the counting weights $M_{ij}$ by an average over time:

$$
\sum_{i,j}M_{ij}\,e^{j(\theta_j-\theta_i)}
\quad\text{becomes}\quad
\frac{1}{T-1}\sum_t e^{j(\phi(t+1)-\phi(t))}.
$$

Write $X_t=A_t e^{j\phi(t)}$ with $A_t=|X_t|$. Weight transition $t$ by $A_t A_{t+1}$ (loud frames count more) and normalize by the same weights:

$$
\frac{\sum_t A_t A_{t+1}\,e^{j(\phi(t+1)-\phi(t))}}{\sum_t A_t A_{t+1}}
=\frac{\sum_t X_t X_{t+1}^{*}}{\sum_t A_t A_{t+1}}.
$$

If $A_{t+1}=A_t$ (constant amplitude), the denominator equals $\sum_t |X_t|^2$, which is the code:

$$
C=\frac{\bigl|\sum_t X_t X_{t+1}^{*}\bigr|}{\sum_t |X_t|^2}.
$$

If amplitude changes slowly, $A_{t+1}\approx A_t$ and the two denominators are close. That is the only approximation. Everything up to the discrete $C_{\mathrm{coh}}$ is an equality.

**Summary.** $p(d)$ is what fraction of steps jumped by $d$. $C_0$ is the largest of those fractions (when every state is used equally). $C_r$ is the largest sum of neighboring fractions. $C_{\mathrm{coh}}$ is the length of the vector sum of those fractions placed on a circle. The code is that last number, with continuous phase and a loudness weight.

## Setup

In [1]:
import os
import librosa
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import utils as ut

In [2]:
fs = 128000
nperseg = 8192
nfft = None
overlap = 0.0
window = "hanning"
dc = 150
crop_freq = 8000
norm_size = 5
f = 632  # Hz, same tonal marker as example.ipynb

## Function definitions

The first block is the discrete graph scores from §§1–6. The second block is the implemented STFT score from §8.

In [3]:
def circular_distance(a, b, Q):
    """Shorter number of steps between states a and b on a circle of Q states."""
    d = np.abs(np.asarray(a) - np.asarray(b))
    return np.minimum(d, Q - d)


def transition_counts(q, Q):
    """N[i, j] = number of times state i is followed by state j."""
    q = np.asarray(q, dtype=int)
    N = np.zeros((Q, Q), dtype=int)
    for i, j in zip(q[:-1], q[1:]):
        N[i, j] += 1
    return N


def row_weights(N):
    """w[i, j] = N[i, j] / N[i]. Empty rows stay 0."""
    N = np.asarray(N, dtype=float)
    row_sum = N.sum(axis=1, keepdims=True)
    return np.divide(N, row_sum, out=np.zeros_like(N), where=row_sum > 0)


def joint_weights(N):
    """M[i, j] = N[i, j] / (total number of transitions)."""
    N = np.asarray(N, dtype=float)
    total = N.sum()
    if total <= 0:
        return np.zeros_like(N)
    return N / total


def hop_histogram(M):
    """p[d] = fraction of transitions whose jump size is d."""
    Q = M.shape[0]
    p = np.zeros(Q, dtype=float)
    for i in range(Q):
        for d in range(Q):
            p[d] += M[i, (i + d) % Q]
    return p


def cyclicity_C_r(w, r):
    """Paper C_r(G): best target jump Δ, counting hops within circular distance r."""
    Q = w.shape[0]
    scores = []
    for delta in range(Q):
        acc = 0.0
        for i in range(Q):
            target = (i + delta) % Q
            for j in range(Q):
                if circular_distance(j, target, Q) <= r:
                    acc += w[i, j]
        scores.append(acc / Q)
    return float(np.max(scores))


def graph_coherence(M):
    """C_coh(G) = |sum_d p(d) exp(j 2π d / Q)| = |sum_{i,j} M_ij exp(j (θ_j - θ_i))|."""
    p = hop_histogram(M)
    Q = p.size
    d = np.arange(Q)
    return float(np.abs(np.sum(p * np.exp(1j * 2 * np.pi * d / Q))))


def phase_coherence_score(x, fs, nperseg, percent_overlap, window,
                          remove_dc=None, crop_freq=None, nfft=None):
    """
    Implemented score, one value per frequency bin:
        C = |mean_t X_t conj(X_{t+1})| / mean_t |X_t|^2
    X is the complex STFT (same window / hop / crop as utils.calc_spectrogram).
    Constant amplitude makes this equal to |mean exp(j Δφ)|; slow AM is the only approximation.
    """
    F, T, Sxx, Phase, X = ut.calc_spectrogram(
        x, fs, nperseg, percent_overlap, nfft=nfft, window=window,
        remove_dc=remove_dc, crop_freq=crop_freq, logscale=False, return_complex=True,
    )
    if X.shape[1] < 2:
        return np.zeros(X.shape[0]), F
    a, b = X[:, :-1], X[:, 1:]
    num = np.abs(np.mean(a * np.conj(b), axis=1))
    den = np.mean(np.abs(a) ** 2, axis=1) + 1e-30
    return num / den, F

## Test on one file

Same recording, STFT settings, and tonal marker as the example cell in `example.ipynb` (`data/example/dpv1_1m.wav`, $f=632$ Hz).

In [4]:
ROOT = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
file_path = os.path.join(ROOT, "data", "example", "dpv1_1m.wav")
data, _ = librosa.load(file_path, sr=fs)

F, T, Sxx, phase = ut.calc_spectrogram(
    data, fs, nperseg=nperseg, percent_overlap=overlap, nfft=nfft,
    window=window, remove_dc=dc, crop_freq=crop_freq,
)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=norm_size)
coherence, F_coh = phase_coherence_score(
    data, fs, nperseg, overlap, window, remove_dc=dc, crop_freq=crop_freq, nfft=nfft,
)

k = ut.nearest_frequency_index(F_coh, f)
print(f"file: {file_path}")
print(f"duration: {len(data) / fs:.2f} s,  bins: {len(F_coh)},  hop frames: {Sxx.shape[1]}")
print(f"coherence at {F_coh[k]:.2f} Hz (marker {f} Hz): {coherence[k]:.4f}")
print(f"coherence max {coherence.max():.4f} at {F_coh[np.argmax(coherence)]:.2f} Hz")

_ds = max(1, Sxx.shape[1] // 800)
zmin, zmax = np.nanpercentile(Sxx, [5, 99])

fig = make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.5, 0.25, 0.25],
    column_titles=["Spectrogram", "Spectrum", "Coherence"],
    shared_yaxes=True,
    horizontal_spacing=0.06,
)
fig.add_trace(
    go.Heatmap(
        z=Sxx[:, ::_ds], x=T[::_ds], y=F,
        colorscale="Viridis", zmin=zmin, zmax=zmax,
        showscale=False, hoverinfo="skip",
    ),
    row=1, col=1,
)
fig.add_trace(go.Scatter(x=pxx, y=F, mode="lines", line=dict(color="#1f77b4", width=1.5), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=coherence, y=F_coh, mode="lines", line=dict(color="#1f77b4", width=1.5), showlegend=False), row=1, col=3)
for col in range(1, 4):
    fig.add_hline(y=f, line_dash="dot", line_color="white" if col == 1 else "crimson", line_width=1, row=1, col=col)

fig.update_layout(height=640, width=1100, font=dict(size=12), margin=dict(t=50, b=60, l=70, r=20), plot_bgcolor="white")
fig.update_yaxes(title_text="Frequency (Hz)", range=[float(F.min()), float(F.max())], row=1, col=1)
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.update_xaxes(title_text="Norm. power", row=1, col=2)
fig.update_xaxes(title_text="C", row=1, col=3)
fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)", zeroline=False)
fig.show()

file: /Users/itamarmerfeld/Projects/Thesis/code/data/example/dpv1_1m.wav
duration: 60.00 s,  bins: 503,  hop frames: 937
coherence at 625.00 Hz (marker 632 Hz): 0.8269
coherence max 0.8446 at 640.62 Hz
